# Week 8 — JavaScript & HTML5 Canvas: Reference Values

**Physics 311 — Block 3: Browser Port**

---

## Why this is a *Python* notebook for a *JavaScript* week

Week 8's code runs in a browser, so it can't execute in a Jupyter Python kernel.
But Lab 8's headline requirement is this:

> **Your JavaScript must produce the same numbers as your Python.**

That makes the most useful notebook for this week an **answer key** — a verified set of
reference values your JavaScript port has to reproduce, plus the JavaScript source
in copy-paste form next to each one.

Every number printed below was produced by running this notebook. Open your browser
console, run the matching JS, and compare.

| Agreement | Verdict |
|---|---|
| Identical to ~1e-12 | Correct port |
| Differs in the 3rd–6th decimal | **Bug** — usually step count, or `dt` mismatch |
| Differs in the 1st decimal | **Bug** — usually an integrator or sign error |

---

## How to use this notebook

1. **Run All.** Record the printed reference values.
2. Build `vec2.js`, `main.js` etc. following the lesson (JS source is in the markdown
   cells below — it is the same code as the lesson page).
3. Open your page with **Live Server**, open DevTools → Console.
4. Fill in the Lab 8 agreement table (§6) with your JS output beside these numbers.

<div style="border-left:4px solid #dc3545;padding:12px 16px">
<b>&#9888;&#65039; A floating-point promise.</b> Both Python and JavaScript use IEEE-754
double precision. The <i>same</i> arithmetic in the <i>same</i> order gives
<i>bit-identical</i> results. So agreement to 1e-12 is not luck — it is the expected
outcome, and any larger discrepancy is a real bug you can find.
</div>

---
# 1 &nbsp; `Vec2`: Replacing `np.array([x, y])`

Your Block 1–2 engine used 2-element NumPy arrays. NumPy's value is vectorizing over
*large* arrays; for 2-element vectors it is pure overhead. In the browser you write a
tiny class instead — and it will be **faster** than NumPy was.

### The JavaScript (`engine/vec2.js`)

```javascript
export class Vec2 {
  constructor(x = 0, y = 0) { this.x = x; this.y = y; }

  // --- allocating ops: return a NEW Vec2 (safe, use by default) ---
  add(v)        { return new Vec2(this.x + v.x, this.y + v.y); }
  sub(v)        { return new Vec2(this.x - v.x, this.y - v.y); }
  mul(s)        { return new Vec2(this.x * s,   this.y * s);   }
  dot(v)        { return this.x * v.x + this.y * v.y; }
  lengthSq()    { return this.x * this.x + this.y * this.y; }
  length()      { return Math.hypot(this.x, this.y); }
  clone()       { return new Vec2(this.x, this.y); }

  normalized() {
    const L = this.length();
    return L === 0 ? new Vec2(0, 0) : new Vec2(this.x / L, this.y / L);
  }

  // --- in-place ops: mutate THIS, allocate nothing (use in hot loops) ---
  addTo(v)          { this.x += v.x;     this.y += v.y;     return this; }
  addScaled(v, s)   { this.x += v.x * s; this.y += v.y * s; return this; }
  set(x, y)         { this.x = x;        this.y = y;        return this; }
  zero()            { this.x = 0;        this.y = 0;        return this; }
}
```

<div style="border-left:4px solid #28a745;padding:12px 16px">
<b>&#128300; Why both allocating and in-place versions?</b> <code>a.add(b)</code> creates
garbage every call. At 60 fps with 50 bodies and 4 forces that is ~12,000 objects per
second for the garbage collector to clean up, which shows up as periodic frame stutter.
Use the readable allocating form everywhere <i>except</i> inside the per-step force and
integrator loops, where <code>addScaled</code> costs nothing.
</div>

### The self-test Lab 8 Part A requires

```javascript
import { Vec2 } from "./engine/vec2.js";

const a = new Vec2(3, 4);
const b = new Vec2(1, -2);

console.log("a.add(b)       =", a.add(b));              // Vec2 {x: 4, y: 2}
console.log("a.sub(b)       =", a.sub(b));              // Vec2 {x: 2, y: 6}
console.log("a.mul(2)       =", a.mul(2));              // Vec2 {x: 6, y: 8}
console.log("a.dot(b)       =", a.dot(b));              // -5
console.log("a.length()     =", a.length());            // 5
console.log("a.lengthSq()   =", a.lengthSq());          // 25
console.log("a.normalized() =", a.normalized());        // Vec2 {x: 0.6, y: 0.8}
console.log("|normalized|   =", a.normalized().length());// 1
```

Run the next cell for the Python values these must match.

In [ ]:
import math

# ---- Python reference for the Vec2 self-test ----
# Deliberately written without NumPy, so the arithmetic matches your JS
# operation-for-operation.

a = (3.0, 4.0)
b = (1.0, -2.0)

def v_add(p, q):  return (p[0] + q[0], p[1] + q[1])
def v_sub(p, q):  return (p[0] - q[0], p[1] - q[1])
def v_mul(p, s):  return (p[0] * s,    p[1] * s)
def v_dot(p, q):  return p[0] * q[0] + p[1] * q[1]
def v_lensq(p):   return v_dot(p, p)
def v_len(p):     return math.hypot(p[0], p[1])
def v_norm(p):
    L = v_len(p)
    return (0.0, 0.0) if L == 0 else (p[0] / L, p[1] / L)

print("=== Vec2 self-test: reference values ===")
print(f"a                 = {a}")
print(f"b                 = {b}")
print(f"a.add(b)          = {v_add(a, b)}")
print(f"a.sub(b)          = {v_sub(a, b)}")
print(f"a.mul(2)          = {v_mul(a, 2)}")
print(f"a.dot(b)          = {v_dot(a, b):.10f}")
print(f"a.length()        = {v_len(a):.10f}")
print(f"a.lengthSq()      = {v_lensq(a):.10f}")
print(f"a.normalized()    = ({v_norm(a)[0]:.10f}, {v_norm(a)[1]:.10f})")
print(f"|a.normalized()|  = {v_len(v_norm(a)):.15f}")

print()
print("Chosen on purpose: the 3-4-5 triangle makes every value exact in")
print("binary floating point, so ANY disagreement is a real bug, not roundoff.")

---
# 2 &nbsp; The Camera: Physics Coordinates &rarr; Pixels

This is the single most common source of "my canvas is blank" and "everything is
upside down." Canvas puts the origin at the **top-left** with **y increasing
downward**. Physics puts it wherever you like with **y increasing upward**.

<div style="border-left:4px solid #17a2b8;padding:12px 16px">
<b>&#128161; Put the flip in exactly one place.</b> One <code>Camera</code> class owns
the transform. If y-flipping is scattered through your drawing code, you will spend an
evening hunting a sign.
</div>

### The JavaScript (`render/camera.js`)

```javascript
export class Camera {
  /**
   * @param {number} scale  pixels per meter
   * @param {number} originX  canvas x-pixel of physics x = 0
   * @param {number} originY  canvas y-pixel of physics y = 0
   */
  constructor(scale, originX, originY) {
    this.scale = scale;
    this.originX = originX;
    this.originY = originY;
  }

  // physics (m) -> canvas (px).  Note the MINUS on y: that is the flip.
  toScreenX(x) { return this.originX + x * this.scale; }
  toScreenY(y) { return this.originY - y * this.scale; }

  // canvas (px) -> physics (m).  The inverse, needed for mouse input in Week 10.
  toWorldX(px) { return (px - this.originX) / this.scale; }
  toWorldY(py) { return (this.originY - py) / this.scale; }

  // a length (m) -> a length (px); no origin, no flip
  toScreenLen(L) { return L * this.scale; }
}
```

### Using it

```javascript
const canvas = document.getElementById("sim");
const ctx = canvas.getContext("2d");

// 20 px per metre, origin 50 px from the left, 50 px up from the bottom
const cam = new Camera(20, 50, canvas.height - 50);

// draw a 0.3 m-radius ball at physics position (2, 3)
ctx.beginPath();
ctx.arc(cam.toScreenX(2), cam.toScreenY(3), cam.toScreenLen(0.3), 0, Math.PI * 2);
ctx.fillStyle = "#ff8200";
ctx.fill();
```

The next cell verifies the round trip and gives you numbers to check against.

In [ ]:
# ---- Camera reference: a 800x600 canvas, 20 px/m, origin at (50, 550) ----
CANVAS_W, CANVAS_H = 800, 600
SCALE   = 20.0
ORIGIN_X, ORIGIN_Y = 50.0, CANVAS_H - 50.0      # 550.0

def to_screen_x(x):  return ORIGIN_X + x * SCALE
def to_screen_y(y):  return ORIGIN_Y - y * SCALE          # the flip
def to_world_x(px):  return (px - ORIGIN_X) / SCALE
def to_world_y(py):  return (ORIGIN_Y - py) / SCALE

print("=== Camera: scale = 20 px/m, origin = (50, 550) on an 800x600 canvas ===")
print(f"{'physics (m)':>16}{'-> screen (px)':>22}")
for pt in [(0, 0), (2, 3), (10, 0), (0, 5), (37.5, 27.5)]:
    print(f"{str(pt):>16}   ->   ({to_screen_x(pt[0]):7.2f}, {to_screen_y(pt[1]):7.2f})")

print()
print("Sanity checks that catch the classic bugs:")
print(f"  physics y=0  lands at py = {to_screen_y(0):.1f}   "
      f"(must be NEAR THE BOTTOM of 600, not 0)")
print(f"  physics y=5  lands at py = {to_screen_y(5):.1f}   "
      f"(must be SMALLER than y=0's pixel)")
print(f"  0.3 m radius -> {0.3 * SCALE:.1f} px")

print()
print("Round-trip test (toWorld(toScreen(p)) must return p):")
max_err = 0.0
for x, y in [(0, 0), (2, 3), (10, 0), (0, 5), (37.5, 27.5), (-3.25, 1.125)]:
    bx = to_world_x(to_screen_x(x))
    by = to_world_y(to_screen_y(y))
    max_err = max(max_err, abs(bx - x), abs(by - y))
print(f"  max round-trip error = {max_err:.3e}   (must be 0.0 exactly)")

print()
print("Visible world bounds for this camera:")
print(f"  x from {to_world_x(0):.2f} m to {to_world_x(CANVAS_W):.2f} m")
print(f"  y from {to_world_y(CANVAS_H):.2f} m to {to_world_y(0):.2f} m")

---
# 3 &nbsp; The Integrators, Ported

Here is the whole point of Block 3 in one observation: **these are the same lines you
wrote in Python.** Semicolons, `let`, and `Math.` replace indentation, `=`, and `np.`.
Nothing physical changes.

| Python (Block 1–2) | JavaScript (Block 3) |
|---|---|
| `v += a * dt` | `b.vel.addScaled(b.acc, dt);` |
| `x += v * dt` | `b.pos.addScaled(b.vel, dt);` |
| `np.sqrt(2)` | `Math.sqrt(2)` |
| `0.5 * m * v**2` | `0.5 * m * v ** 2` |

### The JavaScript (`engine/integrators.js`)

```javascript
export class EulerCromer {
  step(world, dt) {
    for (const b of world.bodies) {
      if (b.fixed) continue;
      b.acc.set(b.force.x * b.invMass, b.force.y * b.invMass);
      b.vel.addScaled(b.acc, dt);     // velocity FIRST
      b.pos.addScaled(b.vel, dt);     // then position, with the NEW velocity
    }
  }
}

export class VelocityVerlet {
  step(world, dt) {
    // 1. half-kick + drift, using the acceleration already stored on the body
    for (const b of world.bodies) {
      if (b.fixed) continue;
      b.pos.x += b.vel.x * dt + 0.5 * b.acc.x * dt * dt;
      b.pos.y += b.vel.y * dt + 0.5 * b.acc.y * dt * dt;
    }
    // 2. recompute forces at the NEW positions
    world.clearForces();
    world.applyForces();
    // 3. finish the velocity with the average of old and new acceleration
    for (const b of world.bodies) {
      if (b.fixed) continue;
      const axNew = b.force.x * b.invMass;
      const ayNew = b.force.y * b.invMass;
      b.vel.x += 0.5 * (b.acc.x + axNew) * dt;
      b.vel.y += 0.5 * (b.acc.y + ayNew) * dt;
      b.acc.set(axNew, ayNew);        // carry it forward — this is the trick
    }
  }
}
```

<div style="border-left:4px solid #dc3545;padding:12px 16px">
<b>&#9888;&#65039; The #1 port bug: <code>let</code> vs. reference semantics.</b>
In Python, <code>r = body.pos</code> then <code>r += v*dt</code> on a NumPy array mutates
the body. In JavaScript, <code>let r = body.pos</code> copies the <i>reference</i>, so
<code>r = r.add(...)</code> silently rebinds your local variable and the body
<b>never moves</b>. Symptom: everything sits perfectly still. Use the in-place
<code>addScaled</code>, or assign back explicitly.
</div>

### The test case

A projectile, $v_0 = 25$ m/s at $45°$, $DT = 1/240$ s, run to exactly $t = 2.0$ s
(480 steps). Gravity only, so there is an exact answer to compare against.

In [ ]:
import math

g, m = 9.81, 1.0
DT = 1.0 / 240.0

def simulate(method, dt, t_end, x0=0.0, y0=0.0, vx0=0.0, vy0=0.0):
    """Port-reference projectile. Mirrors the JS line-for-line (no NumPy)."""
    x, y, vx, vy = x0, y0, vx0, vy0
    E0 = 0.5 * m * (vx * vx + vy * vy) + m * g * y
    Emin = Emax = E0
    t, n = 0.0, 0
    while t < t_end - 1e-12:
        ax, ay = 0.0, -g
        if method == "euler-cromer":
            vx += ax * dt;  vy += ay * dt
            x  += vx * dt;  y  += vy * dt
        elif method == "verlet":
            x += vx * dt + 0.5 * ax * dt * dt
            y += vy * dt + 0.5 * ay * dt * dt
            ax2, ay2 = 0.0, -g                 # constant force: same acceleration
            vx += 0.5 * (ax + ax2) * dt
            vy += 0.5 * (ay + ay2) * dt
        t += dt;  n += 1
        E = 0.5 * m * (vx * vx + vy * vy) + m * g * y
        Emin, Emax = min(Emin, E), max(Emax, E)
    return dict(x=x, y=y, vx=vx, vy=vy, t=t, n=n, E0=E0,
                E=0.5 * m * (vx * vx + vy * vy) + m * g * y,
                band=(Emax - Emin) / E0)

v0, th = 25.0, math.radians(45.0)
VX0, VY0 = v0 * math.cos(th), v0 * math.sin(th)

print("=== Projectile port reference ===")
print(f"v0 = 25 m/s at 45 deg,  DT = 1/240 = {DT:.12f} s,  t_end = 2.0 s")
print(f"vx0 = {VX0:.12f}   vy0 = {VY0:.12f}")
print(f"(both = 25/sqrt(2); if your JS differs here, it is a degrees/radians bug)")
print()

results = {}
for meth in ("euler-cromer", "verlet"):
    r = simulate(meth, DT, 2.0, vx0=VX0, vy0=VY0)
    results[meth] = r
    print(f"-- {meth}   ({r['n']} steps, t = {r['t']:.10f}) --")
    print(f"   x      = {r['x']:.12f}")
    print(f"   y      = {r['y']:.12f}")
    print(f"   vx     = {r['vx']:.12f}")
    print(f"   vy     = {r['vy']:.12f}")
    print(f"   E0     = {r['E0']:.12f}")
    print(f"   E      = {r['E']:.12f}")
    print(f"   dE/E0  = {(r['E'] - r['E0']) / r['E0']:+.6e}")
    print()

t = 2.0
EX, EY, EVY = VX0 * t, VY0 * t - 0.5 * g * t * t, VY0 - g * t
print("-- exact --")
print(f"   x      = {EX:.12f}")
print(f"   y      = {EY:.12f}")
print(f"   vy     = {EVY:.12f}")

### Read those two rows as physics, not just as a checksum

**Velocity Verlet matches the exact answer to 12 decimals.** That is not a coincidence
and it is not Verlet being "accurate" — for a *constant* force, Verlet's position update
$r + v\,\Delta t + \tfrac12 a\,\Delta t^2$ **is** the exact kinematic equation. A
projectile cannot measure Verlet's error at all.

**Euler-Cromer's y is low by exactly 0.040875 m**, and the next cell shows that this is
not a mystery either — it is the closed-form prediction from the Week 2 §5.5 insert:

$$\delta y(t) = -\tfrac{1}{2} g\, t\, \Delta t$$

So the first thing you port reproduces, to 14 digits, a formula you derived in Week 2.
That is a stronger validation of your port than any plot.

In [ ]:
r_ec = results["euler-cromer"]

dy_measured  = r_ec["y"] - EY
dy_predicted = -0.5 * g * 2.0 * DT

print("=== Euler-Cromer position error vs. the Week 2 §5.5 prediction ===")
print(f"  measured  y_EC - y_exact = {dy_measured:.15f} m")
print(f"  predicted -0.5*g*t*dt    = {dy_predicted:.15f} m")
print(f"  difference               = {abs(dy_measured - dy_predicted):.3e} m")
print()

dE_measured  = r_ec["E"] - r_ec["E0"]
dE_predicted = -0.5 * m * g * g * 2.0 * DT

print("  and the energy error, which is ALL potential (velocity is exact):")
print(f"  measured  E - E0         = {dE_measured:.12f} J")
print(f"  predicted -0.5*m*g^2*t*dt= {dE_predicted:.12f} J")
print(f"  difference               = {abs(dE_measured - dE_predicted):.3e} J")
print()
print(f"  Verlet, same run:  dE/E0 = {(results['verlet']['E'] - results['verlet']['E0']) / results['verlet']['E0']:+.3e}")
print()
print("Both velocity columns are IDENTICAL in the two methods above --")
print("exactly as §5.5 argued: with a constant force, velocity is exact in both,")
print("and the entire discrepancy lives in the position update.")

---
# 4 &nbsp; The Honest Test: An Oscillator

The projectile is a degenerate test case (Week 2 §5.5). To actually distinguish the
integrators your port needs a force that **depends on position**, which switches the
error-feedback loop back on.

Same SHO as Lab 3: $m = k = 1$, so $\omega_0 = 1$ and $T = 2\pi$. Released from
$x_0 = 1$, $v_0 = 0$, run **10 full periods** at $DT = 1/240$.

```javascript
// engine/forces.js
export class Spring {
  constructor(body, anchor, k, restLength = 0) {
    this.body = body; this.anchor = anchor;
    this.k = k; this.restLength = restLength;
  }
  apply(world) {
    const d = this.body.pos.sub(this.anchor);
    const L = d.length();
    if (L === 0) return;
    const stretch = L - this.restLength;
    const f = d.mul(-this.k * stretch / L);   // Hooke, along the line
    this.body.force.addTo(f);
  }
  potentialEnergy(world) {
    const stretch = this.body.pos.sub(this.anchor).length() - this.restLength;
    return 0.5 * this.k * stretch * stretch;
  }
}
```

In [ ]:
import math

K, M = 1.0, 1.0
OMEGA0 = math.sqrt(K / M)
PERIOD = 2 * math.pi / OMEGA0

def sho(method, dt, n_periods=10):
    t_end = n_periods * PERIOD
    x, v = 1.0, 0.0
    E0 = 0.5 * K * x * x + 0.5 * M * v * v
    Emin = Emax = E0
    t, n = 0.0, 0
    while t < t_end - 1e-12:
        a = -K * x / M
        if method == "euler":
            xn = x + v * dt
            vn = v + a * dt
            x, v = xn, vn
        elif method == "euler-cromer":
            v += a * dt
            x += v * dt
        elif method == "verlet":
            x += v * dt + 0.5 * a * dt * dt
            a2 = -K * x / M
            v += 0.5 * (a + a2) * dt
        t += dt;  n += 1
        E = 0.5 * K * x * x + 0.5 * M * v * v
        Emin, Emax = min(Emin, E), max(Emax, E)
    return dict(x=x, v=v, n=n, E0=E0,
                E=0.5 * K * x * x + 0.5 * M * v * v,
                worst=max(abs(Emax - E0), abs(Emin - E0)) / E0)

print("=== SHO port reference: m = k = 1, x0 = 1, v0 = 0 ===")
print(f"omega0 = {OMEGA0:.1f}   T = {PERIOD:.12f} s   10T = {10 * PERIOD:.12f} s")
print(f"DT = 1/240,  exact answer after 10 periods:  x = 1.0,  v = 0.0")
print()
print(f"{'method':<14}{'steps':>7}{'x':>16}{'v':>16}{'max|dE|/E0':>14}")
for meth in ("euler", "euler-cromer", "verlet"):
    r = sho(meth, DT)
    print(f"{meth:<14}{r['n']:>7}{r['x']:>16.12f}{r['v']:>16.12f}{r['worst']:>14.3e}")

print()
print("Now the integrators separate by FIVE orders of magnitude:")
print("  Euler        29.9%  energy error  -- amplitude visibly grows")
print("  Euler-Cromer  0.21% -- bounded, no drift")
print("  Verlet        0.0004% -- bounded and 2nd-order small")
print()
print("Euler's x = 1.1399 after 10 periods instead of 1.0: the oscillator")
print("has been pumped 14% wider by the integrator alone. THIS is the")
print("comparison that belongs in your Lab 8 write-up, not the projectile.")

---
# 5 &nbsp; The Fixed-Timestep Accumulator

This is the one genuinely new idea in Week 8. In Python you chose `dt` and looped.
In the browser, `requestAnimationFrame` hands you a frame time you **do not control** —
60 Hz, 144 Hz, 30 Hz on a throttled laptop, or a 2-second freeze when the user switches
tabs.

<div style="border-left:4px solid #17a2b8;padding:12px 16px">
<b>&#128161; Never step physics by the frame time.</b> Physics advances in fixed
<code>DT</code> chunks; rendering happens whenever the browser is ready. The accumulator
is the bridge: bank the elapsed wall time, spend it in whole <code>DT</code> steps, keep
the change.
</div>

### The JavaScript (`main.js`)

```javascript
const DT = 1 / 240;          // physics timestep, seconds — never changes
const MAX_FRAME = 0.25;      // clamp: never try to catch up more than 0.25 s

let accumulator = 0;
let lastTime = performance.now() / 1000;

function frame(nowMs) {
  const now = nowMs / 1000;
  let frameTime = now - lastTime;
  lastTime = now;

  // THE CLAMP. Without it, a tab-switch hands you frameTime = 30 s,
  // you try 7200 physics steps in one frame, and the page locks up.
  if (frameTime > MAX_FRAME) frameTime = MAX_FRAME;

  accumulator += frameTime;
  while (accumulator >= DT) {
    world.step(DT);          // always the SAME dt
    accumulator -= DT;
  }

  renderer.draw(world);      // draw once per frame, however many steps ran
  requestAnimationFrame(frame);
}
requestAnimationFrame(frame);
```

<div style="border-left:4px solid #28a745;padding:12px 16px">
<b>&#128300; Why <code>DT = 1/240</code> and not <code>1/60</code>?</b> Decoupling is the
point. At 1/240 you take ~4 physics steps per 60 Hz frame, so collisions resolve
accurately and the simulation behaves identically on a 60 Hz and a 144 Hz display.
Pick DT from the <i>physics</i> (your Lab 2/3 convergence studies), never from the
monitor.
</div>

The next cell proves frame-rate independence: three very different frame-time
patterns, identical physics.

In [ ]:
DT_ACC = 1.0 / 240.0
MAX_FRAME = 0.25

def run_accumulator(frame_times, dt=DT_ACC, clamp=MAX_FRAME):
    """Exactly the JS loop above, minus the rendering."""
    acc, sim_t, steps = 0.0, 0.0, 0
    biggest_catchup = 0
    for ft in frame_times:
        ft = min(ft, clamp)
        acc += ft
        this_frame = 0
        while acc >= dt:
            sim_t += dt
            steps += 1
            this_frame += 1
            acc -= dt
        biggest_catchup = max(biggest_catchup, this_frame)
    return sim_t, steps, acc, biggest_catchup

smooth  = [1/60] * 600                        # a clean 60 fps for 10 s
high_hz = [1/144] * 1440                      # 144 Hz display, same 10 s
jittery = [1/40, 1/120] * 300                 # uneven, but the SAME 10 s total

print("=== Frame-rate independence ===")
print(f"DT = 1/240 s, clamp = {MAX_FRAME} s")
print()
print(f"{'frame pattern':<16}{'frames':>8}{'wall(s)':>10}{'steps':>8}"
      f"{'sim_t(s)':>14}{'max steps/frame':>17}")
for name, frames in (("smooth 60 Hz", smooth),
                     ("144 Hz", high_hz),
                     ("jittery", jittery)):
    t, s, a, big = run_accumulator(frames)
    print(f"{name:<16}{len(frames):>8}{sum(frames):>10.4f}{s:>8}{t:>14.10f}{big:>17}")

print()
print("All three land on sim_t = 10.0000000000 after exactly 2400 steps,")
print("despite 60 Hz, 144 Hz, and uneven frames. Same physics, different")

print("number of frames drawn. That is frame-rate independence.")

In [ ]:
# ---- Why the clamp matters: simulate a 2-second tab-switch freeze ----
stall = [1/60] * 300 + [2.0] + [1/60] * 299

t_c, s_c, a_c, big_c = run_accumulator(stall, clamp=MAX_FRAME)
t_u, s_u, a_u, big_u = run_accumulator(stall, clamp=1e9)   # clamp disabled

print("=== One 2-second frame (user switched tabs), with and without the clamp ===")
print(f"{'':<22}{'steps':>8}{'sim_t(s)':>14}{'worst single frame':>20}")
print(f"{'clamped at 0.25 s':<22}{s_c:>8}{t_c:>14.6f}{big_c:>20}")
print(f"{'NO clamp':<22}{s_u:>8}{t_u:>14.6f}{big_u:>20}")
print()
print(f"Without the clamp, one frame must run {big_u} physics steps.")
print("At ~50 bodies that is a visible multi-second page freeze -- and if the")
print("steps take longer than real time to compute, the accumulator never")
print("drains and the tab locks up permanently. This failure mode has a name:")
print("the 'spiral of death'.")
print()
print(f"With the clamp, the worst frame runs {big_c} steps and the simulation")
print(f"simply falls {sum(stall) - t_c:.3f} s behind wall-clock time. Falling behind")
print("is a cosmetic problem; locking up is a broken page. Always clamp.")

---
# 6 &nbsp; The Lab 8 Agreement Table

Lab 8 Part C asks for a table comparing your Python and JavaScript output on identical
initial conditions. Run the next cell, then paste your browser console numbers into the
"JavaScript" column.

The JS that produces the matching numbers:

```javascript
// --- Lab 8 agreement harness: no canvas, no rendering, pure numbers ---
import { Vec2 } from "./engine/vec2.js";

const DT = 1 / 240;
const g  = 9.81;

function projectile(method, dt, tEnd, vx0, vy0) {
  let x = 0, y = 0, vx = vx0, vy = vy0, t = 0, n = 0;
  while (t < tEnd - 1e-12) {
    const ax = 0, ay = -g;
    if (method === "euler-cromer") {
      vx += ax * dt;  vy += ay * dt;
      x  += vx * dt;  y  += vy * dt;
    } else {
      x += vx * dt + 0.5 * ax * dt * dt;
      y += vy * dt + 0.5 * ay * dt * dt;
      vx += 0.5 * (ax + 0)  * dt;
      vy += 0.5 * (ay + -g) * dt;
    }
    t += dt;  n++;
  }
  return { x, y, vx, vy, t, n };
}

const v0 = 25, th = 45 * Math.PI / 180;
for (const m of ["euler-cromer", "verlet"]) {
  const r = projectile(m, DT, 2.0, v0 * Math.cos(th), v0 * Math.sin(th));
  console.log(m, JSON.stringify(r, null, 2));
  // print with full precision:
  console.log(`  x  = ${r.x.toFixed(12)}`);
  console.log(`  y  = ${r.y.toFixed(12)}`);
  console.log(`  vy = ${r.vy.toFixed(12)}`);
}
```

<div style="border-left:4px solid #dc3545;padding:12px 16px">
<b>&#9888;&#65039; Use <code>toFixed(12)</code>, not plain <code>console.log</code>.</b>
JavaScript prints <code>15.735339059327</code> as <code>15.735339059326936</code> or
<code>15.735339059327</code> depending on context. Force the precision so you are
comparing digits, not display formats. And compare the <b>step count</b> first — if
<code>n</code> differs, nothing else will match and the bug is in your loop condition,
not your physics.
</div>

In [ ]:
print("=" * 74)
print("LAB 8 AGREEMENT TABLE -- Python reference column".center(74))
print("=" * 74)
print()
print("Shared setup:  v0 = 25 m/s at 45 deg,  DT = 1/240 s,  t_end = 2.0 s,")
print("               m = 1 kg,  g = 9.81 m/s^2,  gravity only")
print()

rows = []
for meth in ("euler-cromer", "verlet"):
    r = simulate(meth, DT, 2.0, vx0=VX0, vy0=VY0)
    rows += [
        (f"{meth}: steps",  f"{r['n']}"),
        (f"{meth}: t",      f"{r['t']:.12f}"),
        (f"{meth}: x",      f"{r['x']:.12f}"),
        (f"{meth}: y",      f"{r['y']:.12f}"),
        (f"{meth}: vx",     f"{r['vx']:.12f}"),
        (f"{meth}: vy",     f"{r['vy']:.12f}"),
        (f"{meth}: dE/E0",  f"{(r['E'] - r['E0']) / r['E0']:+.6e}"),
    ]

for meth in ("euler", "euler-cromer", "verlet"):
    r = sho(meth, DT)
    rows += [
        (f"SHO {meth}: x(10T)",     f"{r['x']:.12f}"),
        (f"SHO {meth}: max|dE|/E0", f"{r['worst']:.6e}"),
    ]

print(f"{'quantity':<30}{'Python (reference)':>22}{'JavaScript':>22}")
print("-" * 74)
for label, val in rows:
    print(f"{label:<30}{val:>22}{'':>22}")
print("-" * 74)
print()
print("Fill in the right-hand column from your browser console.")
print("Agreement to ~1e-12 on every row = a correct port.")

In [ ]:
# ---- Save the reference column so you can diff it programmatically ----
import json, os

ref = {"setup": {"v0": 25.0, "angle_deg": 45.0, "DT": DT, "t_end": 2.0,
                 "m": 1.0, "g": 9.81, "sho": {"k": 1.0, "m": 1.0,
                 "x0": 1.0, "v0": 0.0, "n_periods": 10}},
       "projectile": {}, "sho": {}}

for meth in ("euler-cromer", "verlet"):
    r = simulate(meth, DT, 2.0, vx0=VX0, vy0=VY0)
    ref["projectile"][meth] = {k: r[k] for k in ("x", "y", "vx", "vy", "t", "n", "E0", "E")}

for meth in ("euler", "euler-cromer", "verlet"):
    r = sho(meth, DT)
    ref["sho"][meth] = {k: r[k] for k in ("x", "v", "n", "E0", "E", "worst")}

with open("lab08_reference.json", "w") as f:
    json.dump(ref, f, indent=2)

print("Wrote lab08_reference.json")
print()
print(json.dumps(ref["projectile"]["verlet"], indent=2))
print()
print("You can fetch this file from your JS page and diff automatically:")
print()
print('  const ref = await (await fetch("./lab08_reference.json")).json();')
print('  const err = Math.abs(myY - ref.projectile.verlet.y);')
print('  console.log(err < 1e-12 ? "PASS" : `FAIL by ${err}`);')
print()
print("That is a real unit test, and it is worth the bonus points in Lab 8.")

---
# 7 &nbsp; The Complete Runnable Page

Everything above is numbers. The actual Week 8 deliverable is a page that *draws*.
The companion artifact **`week8_canvas_complete.html`** is a single self-contained file —
open it directly in a browser, no Live Server needed — containing:

- `Vec2` with the self-test printing to the on-page console
- `Camera` with a labelled grid, tick marks, and a ground line
- Euler-Cromer and Velocity Verlet, switchable live
- The clamped fixed-timestep accumulator with a visible step counter
- A live energy-drift HUD (the Week 9 instrument, in embryo)
- A trail, a velocity vector, and a reset button

<div style="border-left:4px solid #6f42c1;padding:12px 16px">
<b>&#9997;&#65039; Try it:</b> Open that page, set the integrator to Euler-Cromer, and read
the energy drift after 2 seconds. It should be <code>-1.28e-03</code> — the number this
notebook printed in §3. Then switch to Verlet and watch it fall to ~1e-15. You have just
validated a browser simulation against a Python reference, which is the entire skill
Block 3 is testing.
</div>

### Why the file layout matters from here on

Lab 8 can live in one `main.js`. Lab 9 cannot. Start splitting now:

```
labs/lab08/
  index.html
  style.css
  main.js              <- wiring only: build scene, start loop
  engine/
    vec2.js            <- Vec2
    body.js            <- Body
    forces.js          <- Gravity, Drag, Spring
    integrators.js     <- EulerCromer, VelocityVerlet
  render/
    camera.js          <- physics <-> pixels
    renderer.js        <- ALL ctx.* drawing
```

<div style="border-left:4px solid #17a2b8;padding:12px 16px">
<b>&#128161; The layering rule, stated once and enforced all semester.</b> Nothing in
<code>engine/</code> may mention <code>canvas</code>, <code>ctx</code>, or
<code>document</code>. Nothing in <code>render/</code> may modify a body's state.
Keep that boundary and the headless convergence study in Week 13 is twenty lines.
Blur it and you will be untangling a renderer under deadline pressure in Week 12.
</div>

---
# 8 &nbsp; Port Bugs, Ranked by How Often I See Them

| Symptom | Cause | Fix |
|---|---|---|
| **Blank canvas, no errors** | `width`/`height` set in CSS instead of HTML attributes | Set `<canvas width="800" height="600">`; CSS sizing stretches the buffer |
| **Blank canvas, console error** | ES6 modules blocked on `file://` | Open with **Live Server**, not by double-clicking |
| **`Failed to load module script`** | Missing `.js` in an import | `from "./engine/vec2.js"` — the extension is required in browsers |
| **Everything upside down** | y-flip missing or doubled | Flip in `Camera.toScreenY` only, nowhere else |
| **Nothing moves, physics "runs"** | `let p = body.pos; p = p.add(v)` rebinds a local | Use `body.pos.addScaled(body.vel, dt)` |
| **Objects drift slowly off-screen** | `addTo` used where `add` was meant, mutating a shared vector | Never store a `Vec2` you also mutate; `clone()` on assignment |
| **Speed changes with monitor refresh rate** | Stepping by `frameTime` instead of `DT` | Use the accumulator |
| **Page freezes after a tab switch** | No frame-time clamp (spiral of death) | `if (frameTime > 0.25) frameTime = 0.25;` |
| **`NaN` everywhere after one frame** | First frame has `lastTime` undefined, or a zero-length normalize | Initialize `lastTime` before the first `rAF`; guard `L === 0` |
| **Trajectory off by ~4 cm** | Euler-Cromer, working correctly | Not a bug — see §3. Switch to Verlet |
| **Energy drift 100× expected** | Degrees passed where radians expected | `Math.cos` takes **radians**; `Math.PI/180` |

<div style="border-left:4px solid #28a745;padding:12px 16px">
<b>&#128300; The debugging order that works.</b> (1) Read the console — the actual error,
not the first red line. (2) Network tab: did every <code>.js</code> load, or did one 404?
(3) <code>console.log</code> one body's position for the first 3 steps and check it
against this notebook. (4) Only then look at the drawing code. Students lose hours
debugging renderers when the physics never ran.
</div>

---

## Where this goes next

| Week | What gets added | What this notebook already gave you |
|---|---|---|
| **9** | Modules, `World`, force registry, collision pipeline, live HUD | `Vec2`, `Camera`, both integrators, the accumulator |
| **10** | Pointer/keyboard input, control panel, deployment | The layering rule that makes input a `Force` |
| **11–13** | Your own project, validated | A verified Python reference to validate *against* |

<div style="border-left:4px solid #ff8200;padding:12px 16px">
<b>The one sentence to carry out of Week 8:</b> your physics was already correct, and you
just proved it — to twelve decimal places, in a new language, in a browser. Everything
in Blocks 3 and 4 is about the <i>medium</i>. The mechanics is done.
</div>